In [107]:
import numpy as np
from collections import Counter

In [108]:
%store -r raw_data

In [109]:
def create_vocab(words, min_count = 5):
    counts = Counter(words)

    vocab = [word for word, freq in counts.items() if freq >= min_count]

    word_to_id = {word: i for i, word in enumerate(vocab)}

    id_to_word = {i: word for word, i in word_to_id.items()}

    return word_to_id, id_to_word

In [110]:
def words_to_ids(words, word_to_id):
    return [word_to_id[w] for w in words if w in word_to_id]

In [111]:
word_to_id, id_to_word = create_vocab(raw_data, min_count = 5)
data_as_ids = words_to_ids(raw_data, word_to_id)

In [112]:
def subsample_data(data_as_ids, threshold = 1e-5):
    counts = np.bincount(data_as_ids)
    total_words = len(data_as_ids)
    
    freqs = counts / total_words + 1e-9
    
    keep_probs = (np.sqrt(freqs / threshold) + 1) * (threshold / freqs)
    keep_probs = np.clip(keep_probs, 0.0, 1.0)
    
    random_draws = np.random.rand(len(data_as_ids))
    word_probs = keep_probs[data_as_ids]
    
    keep_mask = random_draws < word_probs
    subsampled_data = np.array(data_as_ids)[keep_mask]
    
    return subsampled_data

In [113]:
VOCAB_SIZE = len(word_to_id)
EMBED_SIZE = 100
WINDOW_SIZE = 5
NEGATIVE_SIZE = 5
BATCH_SIZE = 2048
LEARNING_RATE = 0.01
EPOCHS = 10

In [114]:
%run ../models/skipgram_model.ipynb
model = SkipgramModel(VOCAB_SIZE, EMBED_SIZE, LEARNING_RATE)

In [115]:
def get_all_pairs(data_as_ids, window_size):
    data = np.array(data_as_ids)
    all_centers = []
    all_contexts = []
    
    for offset in range(-window_size, window_size + 1):
        if offset == 0: continue
        context = np.roll(data, -offset)
    
        if offset > 0:
            all_centers.append(data[:-offset])
            all_contexts.append(context[:-offset])
        else:
            all_centers.append(data[-offset:])
            all_contexts.append(context[-offset:])
            
    return np.concatenate(all_centers), np.concatenate(all_contexts)

In [116]:
clean_data_ids = subsample_data(data_as_ids, threshold = 1e-5)

In [117]:
full_centers, full_contexts = get_all_pairs(clean_data_ids, WINDOW_SIZE)

In [119]:
for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0
    
    for i in range(0, len(full_centers[:1000000]), BATCH_SIZE):
        c_batch = full_centers[i : i + BATCH_SIZE]
        p_batch = full_contexts[i : i + BATCH_SIZE]
        
        loss = model.train_step(c_batch, p_batch)
        
        steps += 1
        total_loss += loss
        
    print(f"Epoch {epoch+1} finished, avg. loss: {total_loss/steps}")

Epoch 1 finished, avg. loss: 4.1662569775094624
Epoch 2 finished, avg. loss: 4.159243615571314
Epoch 3 finished, avg. loss: 4.146823332895514
Epoch 4 finished, avg. loss: 4.098190795051601
Epoch 5 finished, avg. loss: 3.9032251662810973
Epoch 6 finished, avg. loss: 3.4491442959199374
Epoch 7 finished, avg. loss: 2.901727524140595
Epoch 8 finished, avg. loss: 2.506926748870952
Epoch 9 finished, avg. loss: 2.2701817692282718
Epoch 10 finished, avg. loss: 2.122024247043216
